# Counter quality and redundancy

Compare the current **316-input logistic regression** with extra redundancy features, HP-quality features and both. Same five grouped development folds; old holdout and duplicates excluded. Original 316 inputs and LR settings stay fixed. Default: review saved evidence; enable the flag to repeat four candidates.


In [ ]:
from pathlib import Path
import sys, csv, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/pokemon_challenge").is_dir())
sys.path.insert(0, str(ROOT / "notebooks/leonard"))
RUN_NEW_EXPERIMENT = False
RUN_FOLDER = ROOT / "outputs/leonard/20261103_140058_542958_counter_quality"
if RUN_NEW_EXPERIMENT:
    from run_counter_experiment import run_counters
    RUN_FOLDER = run_counters()
print("Artifacts:", RUN_FOLDER)


## What is new?

A favourable counter has potential own-type effectiveness >1. A safe counter also resists the enemy's own-type options (<1). This is a type-based estimate, not an actual switch/damage guarantee.

**Redundancy (8):** number of favourable/safe counters, weakest counter count, at least two counters, uncovered enemies, exactly one safe counter.

**HP quality (8):** healthiest favourable/safe counter, weakest healthiest counter, healthy-safe coverage (HP≥.5), total counter HP reserves, and enemy-HP-weighted uncovered fraction. Unknown enemy types remain missing; own never-seen members start healthy. Metadata, imputation, selection and scaling are learned inside each training fold.


In [ ]:
with (RUN_FOLDER / "cv_results.csv").open() as stream:
    for row in csv.DictReader(stream):
        print(f"{row['candidate']:23s} {float(row['mean_selected_features']):.0f} inputs  {100*float(row['cv_accuracy']):.3f}%")
with (RUN_FOLDER / "paired_comparison.csv").open() as stream:
    for row in csv.DictReader(stream):
        print(row["candidate"], "delta pp:", round(float(row["delta_pp"]), 3), "folds better:", row["folds_better"], "corrected:", row["corrected_errors"], "introduced:", row["introduced_errors"])


## Decision

Use **HP quality only: 324 inputs, 67.396% mean CV**, compared with 67.259% for the current baseline. Four folds improve, one declines; net 11 more correct predictions among 7,996. Redundancy alone and the combination each score 67.284%.

This is a small exploratory gain, not independent confirmation. The full-data winner retains all previous 316 selected inputs and adds the eight quality features, then fits all 10,000 labels to create 5,000 test predictions. No new holdout score or test labels. These type/HP metrics omit other battle mechanics and may overestimate a counter that is asleep, trapped or otherwise disadvantaged.


In [ ]:
with (RUN_FOLDER / "new_feature_definitions.csv").open() as stream:
    for row in csv.DictReader(stream):
        if row["used_in_final_model"] == "True":
            print(row["feature"] + ": " + row["description"])
print(json.dumps(json.loads((RUN_FOLDER / "validation_report.json").read_text()), indent=2))
print("Submission:", RUN_FOLDER / "submission.csv")
